# 03 — YOLOv8 Object Detection
Run YOLOv8 on KITTI camera images to detect cars, pedestrians, and cyclists.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics opencv-python matplotlib -q

In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

BASE_PATH  = '/content/drive/MyDrive/Project/Kitti/Training'
IMAGE_PATH = os.path.join(BASE_PATH, 'image_2/training/image_2')

model       = YOLO('yolov8n.pt')
sample_id   = sorted(os.listdir(IMAGE_PATH))[0].replace('.png','')
img_path    = f'{IMAGE_PATH}/{sample_id}.png'

results     = model(img_path, conf=0.3, classes=[0,1,2,3,5,7])
boxes       = results[0].boxes.xyxy.cpu().numpy()
scores      = results[0].boxes.conf.cpu().numpy()
class_ids   = results[0].boxes.cls.cpu().numpy()
class_names = {0:'person',1:'bicycle',2:'car',3:'motorcycle',5:'bus',7:'truck'}
colors      = {0:'red',1:'orange',2:'lime',3:'cyan',5:'magenta',7:'yellow'}

print(f'Detections: {len(boxes)}')
for box, score, cls in zip(boxes, scores, class_ids):
    print(f'  {class_names.get(int(cls)):10s}  conf={score:.2f}  box={box.astype(int).tolist()}')

In [ ]:
img     = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(1, figsize=(16,6))
ax.imshow(img_rgb)

for box, score, cls in zip(boxes, scores, class_ids):
    x1,y1,x2,y2 = box
    cls_int = int(cls)
    color   = colors.get(cls_int, 'white')
    name    = class_names.get(cls_int, 'unknown')
    rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-5, f'{name} {score:.2f}', color=color,
            fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))

ax.set_title(f'YOLOv8 Detections — Sample {sample_id}')
ax.axis('off'); plt.tight_layout(); plt.show()